## Questions commerciales

1. Quel produit cumule le plus grand nombre de ventes au total ?

2. Quel article est le plus/moins vendu parmi chacun des "departments" ?

3. Quel produit est généralement vendu avec chacun des articles les plus populaires ?

4. Quelle est la probabilité qu'un acheteur rachète un produit parmi une catégorie dont il a déjà commandé ?

5. Peut-on prédire la date de la prochaine commande d'un client en fonction de la fréquence à laquelle il commande ?

6. Est-ce que le jour et l'heure influence le nombre de vente parmi chaque "departement" ?

7. Certains articles sont-ils recommandés plus d'une fois ?

8. Existe-t-il une corrélation entre le premier et le deuxième article acheté ?

9. Pour quels jour et heure la catégorie alcool comporte le plus de vente ?

10. Peut-on prédire la taille d'un panier pour un client achetant de l'alcool ?

11. Parmi la catégorie "bakery", est-ce que les clients achètent plus de pain ("bread") le matin que de dessert ("bakery desserts") le midi et le soir ?

## Import

In [105]:
import pandas as pd
import os
import re
import numpy as np
import matplotlib.pyplot as plt

## Lecture des CSV

In [106]:
df_order_products = pd.read_table("./datasets-project/datasets/order_products.csv", delimiter=",", low_memory = False)
df_orders = pd.read_table("./datasets-project/datasets/orders.csv", delimiter=",", low_memory = False)
df_products = pd.read_table("./datasets-project/datasets/products.csv", delimiter=",", low_memory= False)
df_aisles = pd.read_table("./datasets-project/datasets/aisles.csv", delimiter=",", low_memory= False)
df_departments = pd.read_table("./datasets-project/datasets/departments.csv", delimiter="," ,low_memory=False)

## Observations

In [107]:
# ---------------- order_product.csv ---------------- #

#for i in range(len(df_order_products["product_id"])):
#    if df_order_products["product_id"].iloc[i] <= 0:
#        print(df_order_products)

print(df_order_products.head())
print(df_order_products.shape)
#print(df_order_products)

# ---------------------------------------------- #

   order_id  product_id  add_to_cart_order  reordered
0         2     33120.0                1.0        1.0
1         2     28985.0                2.0        1.0
2         2      9327.0                3.0        0.0
3         2     45918.0                4.0        1.0
4         2     30035.0                5.0        0.0
(13692886, 4)


In [108]:
# ---------------- orders.csv ---------------- #

print(df_orders.head())
print(df_orders.shape)
#print(df_orders)
df_orders.info()

# ---------------------------------------------- #

   order_id  user_id  order_number  order_dow  order_hour_of_day  \
0   2539329      1.0           1.0        2.0                8.0   
1   2398795      1.0           2.0        3.0                7.0   
2    473747      1.0           3.0        3.0               12.0   
3   2254736      1.0           4.0        4.0                7.0   
4    431534      1.0           5.0        4.0               15.0   

   days_since_prior_order  
0                     NaN  
1                    15.0  
2                    21.0  
3                    29.0  
4                    28.0  
(1444444, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1444444 entries, 0 to 1444443
Data columns (total 6 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   order_id                1444444 non-null  int64  
 1   user_id                 1444413 non-null  float64
 2   order_number            1444421 non-null  float64
 3   order_dow         

In [ ]:
# ---------------- products.csv ---------------- #

print(df_products.head(), "\n")
print(df_products.shape, "\n")
df_products.info()


print("\nNb de product_id inférieur à 1 :", len(df_products[df_products["product_id"] < 1]))
print("Nb de product_id n'étant pas des entiers :", len(df_products[df_products["product_id"].apply(lambda x: isinstance(x, int) == False)]))
if df_products["product_id"].duplicated().any() == True:
    print("Doublons :", df_products[df_products["product_id"].duplicated()], "\n")
else:
    print("Il n'y a aucun product_id en double", "\n")


print("Nb de product_name contenant un '+' :", len(df_products[df_products["product_name"].apply(lambda x: "+" in str(x))]))
print("Nb de product_name contenant des guillemets :", len(df_products[df_products["product_name"].apply(lambda x: str(x).strip().startswith("'") and str(x).strip().endswith("'"))]))
'''
Débuggage 
for k in range(len(df_products.index)):
    if k == 74:
        print(df_products["product_id"].iloc[k], str(df_products["product_name"].iloc[k]).strip()) #product_name étant normalement entouré de guillemets
'''
print("Nb de product_name contenant des caractères spéciaux :", len(df_products[df_products["product_name"].apply(lambda x: re.compile(r'[@_!#$%^&*()<>?/\|}{~:"]').search(str(x)) != None)]))
if df_products["product_name"].duplicated().any() == True:
    print("Doublons :", df_products[df_products["product_name"].apply(lambda x: str(x).lower()).duplicated()], "\n")
else:
    print("Il n'y a aucun product_name en double", "\n")


print("Nb de aisle_id inférieur à 1 :", len(df_products[df_products["aisle_id"] < 1]))
print("Nb de aisle_id supérieur à 134 :", len(df_products[df_products["aisle_id"] > 134]))
print("Nb de aisle_id n'étant pas des entiers :", len(df_products[df_products["aisle_id"].apply(lambda x: isinstance(x, int) == False)]), "\n")


print("Nb de department_id inférieur à 1 :", len(df_products[df_products["department_id"] < 1]))
print("Nb de department_id supérieur à 21 :", len(df_products[df_products["department_id"] > 21]))
print("Nb de department_id n'étant pas des entiers :", len(df_products[df_products["department_id"].apply(lambda x: isinstance(x, int) == False)]))

# ---------------------------------------------- #

   product_id                                       product_name  aisle_id  \
0           1                         Chocolate Sandwich Cookies        61   
1           2                                   All-Seasons Salt       104   
2           3               Robust Golden Unsweetened Oolong Tea        94   
3           4  Smart Ones Classic Favorites Mini Rigatoni Wit...        38   
4           5                          Green Chile Anytime Sauce         5   

   department_id  
0             19  
1             13  
2              7  
3              1  
4             13   

(49688, 4) 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49688 entries, 0 to 49687
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   product_id     49688 non-null  int64 
 1   product_name   49688 non-null  object
 2   aisle_id       49688 non-null  int64 
 3   department_id  49688 non-null  int64 
dtypes: int64(3), object(1)
memory u

In [110]:
# ---------------- Questions commerciales ---------------- #

#Question 11 - Trouver les sous-catégories de 'bakery'
df_products_bakery = df_products[df_products["department_id"] == 3]
aisles_id = list(df_products_bakery["aisle_id"].unique())
aisle_names = []
for aisle_id in aisles_id:
    for k in range(len(df_aisles.index)):
        if df_aisles["aisle_id"].iloc[k] == int(aisle_id):
            if df_aisles["aisle"].iloc[k] not in aisle_names:
                aisle_names.append(df_aisles["aisle"].iloc[k])
print("Les sous catégories de 'bakery' sont :", ", ".join(aisle_names))

# ---------------------------------------------- #

Les sous catégories de 'bakery' sont : tortillas flat bread, bread, breakfast bakery, bakery desserts, buns rolls


#### Interprétation des datasets

- "product.csv" permet de créer les fiches produits en associant à chaque élément une catégorie, "department", puis une sous catégorie, "aisle" : DEPARTEMENT ---> AISLE ---> PRODUCT

- "order.csv", donne l'id du client ainsi que les informations sur la commande sans donner ce que comporte le panier

- "oder_product.csv" ---> fait la liaison entre les commandes effectuées et leur panier, "order_id" fournit les informations de la commande et "product_id" y associe un produit


## Nettoyage

In [111]:
# ---------------- Clean orders ---------------- #

df_orders_clean = df_orders.dropna(subset=['order_id','user_id','order_number','order_dow'])

# ---------------------------------------------- #

# ---------------- Clean order products ---------------- #

df_order_products_clean = df_order_products.dropna()

# ------------------------------------------------------ #

# ---------------- Clean products ---------------- #

df_products_clean = df_products.dropna().drop_duplicates()
df_products_clean["product_name"] = df_products_clean["product_name"].astype("string")
df_products_clean["product_name"] = df_products_clean["product_name"].astype(str).str.replace('"', "",regex=False).str.lower()
df_products_clean["product_name"] = df_products_clean["product_name"].str.lower().drop_duplicates()


# ------------------------------------------------ #

# ---------------- Fusion ---------------- #

# Étape 1:

df_products_clean = df_products_clean.merge(df_aisles, how = "inner", on="aisle_id")
df_products_clean = df_products_clean.merge(df_departments, how = "inner", on="department_id")

# ------ #

# Étape 2:

df_order_products_clean = df_order_products_clean.merge(df_orders_clean, how = "inner", on="order_id")
df_order_products_clean = df_order_products_clean.merge(df_products_clean, how = "inner", on="product_id")

# ------ #

# Étape 3:

df_order_products_clean = df_order_products_clean.drop(["order_id","product_id","department_id","aisle_id"], axis=1)

# ------ #

# ---------------------------------------- #

# ---------------- Print ---------------- #

print(df_order_products_clean)
print(df_order_products_clean.shape)
print(df_order_products_clean.head())

          add_to_cart_order  reordered   user_id  order_number  order_dow  \
0                       1.0        1.0  202279.0           3.0        5.0   
1                       2.0        1.0  202279.0           3.0        5.0   
2                       3.0        0.0  202279.0           3.0        5.0   
3                       4.0        1.0  202279.0           3.0        5.0   
4                       5.0        0.0  202279.0           3.0        5.0   
...                     ...        ...       ...           ...        ...   
13690733                4.0        1.0  100467.0           5.0        2.0   
13690734                5.0        0.0  100467.0           5.0        2.0   
13690735                6.0        1.0  100467.0           5.0        2.0   
13690736                7.0        1.0  100467.0           5.0        2.0   
13690737                8.0        0.0  100467.0           5.0        2.0   

          order_hour_of_day  days_since_prior_order            product_name

## Export des CSV nettoyés

In [112]:
if not os.path.exists("./datasets-cleaned"):
    os.makedirs("./datasets-cleaned")

df_orders_clean.to_csv("./datasets-cleaned/df_orders_clean.csv", index=False)
df_order_products_clean.to_csv("./datasets-cleaned/df_order_products_clean.csv", index=False)
df_products_clean.to_csv("./datasets-cleaned/df_products_clean.csv", index=False)

KeyboardInterrupt: 